# 第 60 课｜P2 工具链总结与完成度审计

本 Notebook 对应「365 天量化金融」第 60 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/60_youtube-2PGtdJ9pXfQ/index.html`
- 环境：`pkm-quant`
- 数据：本地 P2 项目文件、环境版本和产物清单
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

P2 的目标不是记住库名，而是形成可替换研究链：

```text
数据适配 → 存储 → 清洗/特征 → 统计/模型 → 验证 → 报告
```

本节把工具映射到职责，并自动检查 31–60 课和 L1–L6 是否都有文件。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# importlib.metadata 读取已安装版本，hashlib 生成项目清单。
import importlib.metadata as metadata
import hashlib

## 1. 工具选择服从瓶颈类型

总耗时可拆为 `T_total = T_data + T_compute + T_validation + T_report`。先识别最大且可控的部分，再选择 asyncio、Numba、缓存或报告模板。

In [3]:
# 建立课程工具与量化职责映射。
tool_map = pd.DataFrame([
    ("NumPy / pandas", "数值与标签计算", "收益、矩阵、时间对齐"),
    ("Matplotlib / Plotly / mplfinance", "诊断展示", "静态图、交互图、K线"),
    ("yfinance / AkShare adapter", "数据适配", "字段、复权、失败状态"),
    ("SQLite", "单机研究存储", "唯一键、UPSERT、查询"),
    ("SciPy / statsmodels / sklearn", "统计与模型", "厚尾、归因、时序预测"),
    ("asyncio / threads", "I/O 等待", "受控并发获取"),
    ("Numba", "数值热点", "预热后的显式循环"),
    ("pytest / Git / Docker", "质量与复现", "测试、版本、环境"),
    ("Jinja2", "交付", "本地自动月报"),
], columns=["工具", "职责", "量化使用"])
display(tool_map)

,工具,职责,量化使用
0,NumPy / pandas,数值与标签计算,收益、矩阵、时间对齐
1,Matplotlib / Plotly / mplfinance,诊断展示,静态图、交互图、K线
2,yfinance / AkShare adapter,数据适配,字段、复权、失败状态
3,SQLite,单机研究存储,唯一键、UPSERT、查询
4,SciPy / statsmodels / sklearn,统计与模型,厚尾、归因、时序预测
5,asyncio / threads,I/O 等待,受控并发获取
6,Numba,数值热点,预热后的显式循环
7,pytest / Git / Docker,质量与复现,测试、版本、环境
8,Jinja2,交付,本地自动月报


## 2. 自动检查课程和综合实验文件

逐课 Notebook 的主键是课程编号；综合实验主键是 L1–L6。只检查文件存在不等于执行通过，完整验收还要结合 `notebook-validation.json`。

In [4]:
# 检查 30 个逐课 Notebook 和 6 个实验 Notebook 的文件覆盖。
lesson_paths = sorted((PROJECT_ROOT / "notebooks").glob("*.ipynb"))
lab_paths = sorted((PROJECT_ROOT / "labs").glob("*.ipynb"))
lesson_numbers = {int(path.name.split("_", 1)[0]) for path in lesson_paths}
lab_ids = {path.name.split("_", 1)[0] for path in lab_paths}
missing_lessons = sorted(set(range(31, 61)) - lesson_numbers)
missing_labs = sorted({f"L{i}" for i in range(1, 7)} - lab_ids)

coverage = pd.Series({
    "lesson_notebooks": len(lesson_paths),
    "lab_notebooks": len(lab_paths),
    "missing_lessons": missing_lessons,
    "missing_labs": missing_labs,
})
display(coverage.to_frame("value"))
assert len(lesson_paths) == 30 and not missing_lessons
assert len(lab_paths) == 6 and not missing_labs

,value
lesson_notebooks,30
lab_notebooks,6
missing_lessons,[]
missing_labs,[]


## 3. 环境版本与 P2 完成清单

清单保存关键依赖版本和所有 Notebook 的 SHA-256。任何 Notebook 内容变化都会反映在下一次清单中。

In [5]:
# 生成 P2 完成清单；不把文件名等同于执行通过。
packages = [
    "numpy", "pandas", "matplotlib", "scipy", "scikit-learn",
    "statsmodels", "numba", "pytest", "jinja2", "plotly", "mplfinance",
    "yfinance", "akshare",
]
versions = {package: metadata.version(package) for package in packages}
notebook_hashes = {
    str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in [*lesson_paths, *lab_paths]
}
validation_path = PROJECT_ROOT / "notebook-validation.json"
validation = json.loads(validation_path.read_text(encoding="utf-8")) if validation_path.exists() else {}
validation_passed = validation.get("status") == "pass" and validation.get("notebook_count") == 36
manifest = {
    "phase": "P2",
    "status": "validated-pass" if validation_passed else "notebooks-built",
    "lessons": len(lesson_paths),
    "labs": len(lab_paths),
    "validation_report": validation_path.name if validation_passed else None,
    "packages": versions,
    "notebook_sha256": notebook_hashes,
}
manifest_path = OUTPUT_DIR / "60_p2_completion_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.Series({**versions, "full_validation": validation_passed}, name="version/status").to_frame())
assert len(notebook_hashes) == 36

,version/status
numpy,2.4.6
pandas,3.0.3
matplotlib,3.11.0
scipy,1.17.1
scikit-learn,1.9.0
statsmodels,0.14.6
numba,0.66.0
pytest,9.1.1
jinja2,3.1.6
plotly,6.9.0


## 方法边界与复用结论


- 更多工具会增加兼容和认知成本；稳定接口比工具数量重要。
- 性能优化不能优先于无未来信息、数据正确性和成本建模。
- P2 完成代表工具链学习闭环，不代表已经具备生产实盘系统。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。